# Fan-out - Testing Notebook

Checks how compound questions (two+ parts across different lanes) are handled. The router
detects `compound`, splits it into sub-tasks, the graph runs each part **in parallel**, and the
answers are combined.

## Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from langchain_core.messages import HumanMessage
from src.agents.router import classify
from src.graph import ask

## Part 1 - Decomposition (one router call each)

`classify` should mark cross-lane questions `compound` and split them, but leave single-lane
questions alone (even ones with the word "and").

In [2]:
def show_route(q):
    r = classify([HumanMessage(q)])
    subs = [(s.intent, s.question) for s in r.subtasks]
    print(f"[{r.intent}]  {q}")
    for intent, question in subs:
        print(f"     - ({intent}) {question}")
    print()

# Compound (different lanes) -> should split
show_route("Who are my top tenants, and is anything unusual in the numbers?")
show_route("What's my 2024 P&L, and chart my 2025 revenue by month?")

# NOT compound (single lane) -> should stay one intent, no subtasks
show_route("What is the total P&L for 2024 and 2025?")
show_route("Is anything unusual?")

[compound]  Who are my top tenants, and is anything unusual in the numbers?
     - (analytics) Who are my top tenants by revenue across all properties and all available time periods?
     - (insights) Is anything unusual or anomalous in the financial numbers across all properties and tenants?



[compound]  What's my 2024 P&L, and chart my 2025 revenue by month?
     - (analytics) What is the net P&L for the entire portfolio in 2024?
     - (visualize) Chart revenue by month for the entire portfolio in 2025.



[analytics]  What is the total P&L for 2024 and 2025?



[insights]  Is anything unusual?



## Part 2 - End to end (the PDF's example)

The compound question should come back with BOTH parts answered.

In [3]:
r = ask("Who are my top tenants, and is anything unusual in the numbers?", thread_id="fanout-nb")
print("intent:", r["intent"])
print("both parts present:",
      "Tenant 7" in r["answer"], "and", any(w in r["answer"].lower() for w in ["unusual", "anomal", "discount"]))
print()
print(r["answer"][:900])

intent: compound
both parts present: True and True

**Who are my top tenants by revenue across all properties and all available time periods?**

# Top Tenants by Revenue

Here are your top tenants across all properties and available time periods:

1. **Tenant 7** – $880,535.66
2. **Tenant 14** – $391,490.26
3. **Tenant 11** – $292,531.00
4. **Tenant 13** – $274,344.45
5. **Tenant 3** – $204,788.42
6. **Tenant 12** – $166,887.73
7. **Tenant 15** – $153,179.91
8. **Tenant 16** – $115,145.70
9. **Tenant 18** – $91,925.71
10. **Tenant 2** – $64,323.40

Tenant 7 is your clear revenue leader, generating nearly 2.25 times more revenue than your second-place tenant.

**Is anything unusual or anomalous in the financial numbers across all properties and tenants?**

Several anomalies have been flagged. Here are the most notable findings:

**Key Anomalies:**

1. **Real Estate Taxes (July 2024)** – Sign flip: posted as +1,086.0 instead of the typical 


## Part 3 - Edge cases

- A three-part compound splits into three.
- A single-lane question is never sent through fan-out.

In [4]:
r = ask("Show me the total expenses in 2024, the top tenant, and anything unusual.", thread_id="fanout-3")
print("intent:", r["intent"])
print()
print(r["answer"][:700])

intent: compound

**What are the total expenses for the entire portfolio in 2024?**

The total expenses for the entire portfolio in 2024 are **$1,124,007.19**.

**Which tenant generated the most revenue in 2024?**

Based on the 2024 data, **Tenant 7** generated the most revenue with **$703,009.03**.

**Are there any unusual or anomalous patterns in the 2024 financial data?**

## Summary of Notable 2024 Anomalies

**1. Real Estate Taxes – July 2024 Sign Flip**
   - July showed a positive value of **1,086.0** (a credit), while the typical pattern is negative (around **-4,170**). This is a significant departure and worth investigating—it may indicate a tax refund or adjustment that month.

**2. Rent Discount (Un


## Notes

- **Parallel:** the parts are independent, so the fan-out node runs them in a thread pool - the
  wall-clock time is about the *slowest* part, not the sum.
- **Guardrail against over-splitting:** "P&L for 2024 and 2025" is a single analytics question, so
  it is NOT marked compound.
- **v1 limitation:** a chart part inside a compound is answered as text (charts render one per
  message); parallelising via LangGraph's `Send` API instead of threads is possible future work.